# Tome 12, Chapter 10: Class field theory (Python companion)

Exercises first, solutions at the end. We test cubic reciprocity and Gauss's theorem on thousands of primes, compute Jacobi sums, ask PARI for Hilbert class fields and check the criteria for $x^2+5y^2$, $x^2+14y^2$ and $x^2+xy+6y^2$, find conductors of abelian fields (Kronecker–Weber in practice), check the weight-one modular form $\eta(6z)\eta(18z)$ against the Frobenius of $X^3-2$, and end with the main application of class field theory to computing: the **CM method**, which builds elliptic curves with a prescribed number of points, for cryptography. Requires `cypari2`.

In [ ]:
import math, random, time
from collections import Counter
import cypari2
pari = cypari2.Pari()
pari.allocatemem(10**9, silent=True)
random.seed(10)

## Exercises

**P10.1 (★) Cubic reciprocity, by brute force.** Represent $a+b\omega\in\mathbb Z[\omega]$ as a pair. For a primary prime $\pi$ of norm $p\equiv1\pmod3$, compute $\chi_\pi(\alpha)$ by working in $\mathbb Z[\omega]/\pi\cong\mathbb Z/p$ (send $\omega$ to the root $r$ of $r^2+r+1\equiv0$ with $\pi\mid r-\omega$). Check Theorem 10.20 on all pairs of primary primes of norm below $2000$, and Gauss's theorem (Theorem 10.21) for all $p<10^5$.

**P10.2 (★) Jacobi sums.** For each $p\equiv1\pmod3$ below $500$, compute $J(\chi,\chi)$ exactly in $\mathbb Z[\omega]$ for a cubic character $\chi$ modulo $p$, and check Lemma 10.19: it has norm $p$, it is primary, and it is the prime $\pi$ with $\chi=\chi_\pi$. Read off the representation $4p=L^2+27M^2$ (Gauss) from $J$.

**P10.3 (★★) Hilbert class fields.** With PARI's `quadhilbert`, compute the Hilbert class fields of $\mathbb Q(\sqrt{-5})$, $\mathbb Q(\sqrt{-14})$, $\mathbb Q(\sqrt{-23})$, and check with `rnfconductor` that $K(\sqrt{2\sqrt2-1})/\mathbb Q(\sqrt{-14})$ is unramified. Then check Theorem 10.15 for $n=5,14$ and Exercise 10.7 for all $p<10^5$.

**P10.4 (★★) Kronecker–Weber in practice.** For the abelian fields defined by $X^3-3X+1$, $X^3+X^2-4X+1$, $X^4-5X^2+5$, $X^4+X^3+X^2+X+1$ and $X^2-X+5$, find the conductor (the least $m$ with $K\subseteq\mathbb Q(\zeta_m)$): by Theorem 10.1 only the ramified primes divide $m$; test candidates with `nfroots` or `nfisincl`. Compare the conductor with the discriminant. Then check the Langlands example of Section 10.4: the coefficients $a_p$ of $q\prod(1-q^{6n})(1-q^{18n})$ against the number of roots of $X^3-2$ modulo $p$, for $p<5000$.

**P10.5 (★★★) CS application: the CM method.** To use an elliptic curve $E/\mathbb F_p$ in cryptography one wants $\#E(\mathbb F_p)$ prime. Counting points on a random curve (Schoof–Elkies–Atkin, `ellcard`) costs about half a second at $128$ bits, and one needs dozens of tries. The CM method turns the problem around, using class field theory for an imaginary quadratic field $K=\mathbb Q(\sqrt D)$:
1. Choose a fundamental discriminant $D<0$ with small class number, $D\equiv5\pmod8$, and search for $t,v$ odd with $p=\frac{t^2+|D|v^2}4$ prime and $N=p+1-t$ prime.
2. The roots modulo $p$ of the Hilbert class polynomial $H_D$ (PARI `polclass(D)`, of degree $h(D)$; its roots generate the Hilbert class field of $K$ over $K$, Kronecker's Jugendtraum) are the $j$-invariants of curves with $p+1\pm t$ points. Since $p=\pi\bar\pi$ with $\pi$ principal, $p$ splits completely in the Hilbert class field, so $H_D$ splits into linear factors modulo $p$.
3. Build $E:y^2=x^3+3kx+2k$, $k=\frac j{1728-j}$, check its order, and replace it by its quadratic twist if it has $p+1+t$ points.
Implement it at $128$ bits, verify that $H_D$ splits completely modulo $p$, and compare the time with the naive method (random curves until the order is prime).

## Solutions

### P10.1
**How to find it.** If $\pi=a+b\omega$ has prime norm $p$, then $p\nmid b$ and $\omega\equiv-ab^{-1}\pmod\pi$, so $a+b\omega\mapsto a+br$ with $r\equiv-ab^{-1}\pmod p$ is the isomorphism $\mathbb Z[\omega]/\pi\to\mathbb Z/p$. Then $\chi_\pi(\alpha)$ is read off from $\alpha^{(p-1)/3}\bmod p$, compared with the images $1,r,r^2$ of $1,\omega,\omega^2$. For $\varpi=q$ rational, $\mathbb Z[\omega]/q=\mathbb F_{q^2}$; we only need $\chi_\pi(q)$ and $\chi_q(\pi)$, and the second is computed in $\mathbb F_q[\omega]$ by fast exponentiation.

In [ ]:
def mul(x, y):                     # (a + b w)(c + d w), w^2 = -1 - w
    a, b = x; c, d = y
    return (a*c - b*d, a*d + b*c - b*d)

def powmod_q(x, e, q):             # in Z[w]/q
    r = (1, 0); x = (x[0] % q, x[1] % q)
    while e:
        if e & 1: r = mul(r, x); r = (r[0] % q, r[1] % q)
        x = mul(x, x); x = (x[0] % q, x[1] % q); e >>= 1
    return r

def primary_primes(bound):
    out = []
    for p in range(7, bound, 3):
        if p % 3 == 1 and pari.isprime(p):
            for a in range(-60, 61):
                for b in range(-60, 61):
                    if a*a - a*b + b*b == p and a % 3 == 2 and b % 3 == 0:
                        out.append((a, b, p))
    return out

def chi(pi, alpha):
    a, b, p = pi
    r = -a * pow(b, -1, p) % p
    x = (alpha[0] + alpha[1] * r) % p
    if x == 0: return None
    y = pow(x, (p - 1) // 3, p)
    return {1: 0, r: 1, r * r % p: 2}[y]           # exponent k with chi = w^k

def chi_q(q, alpha):
    y = powmod_q(alpha, (q * q - 1) // 3, q)
    return {(1, 0): 0, (0, 1): 1, ((-1) % q, (-1) % q): 2}[y]

P = [x for x in primary_primes(2000) if abs(x[0]) <= 60]
bad = sum(1 for x in P for y in P if x[2] != y[2] and chi(x, y[:2]) != chi(y, x[:2]))
Q = [q for q in range(5, 200, 3) if pari.isprime(q)]
bad += sum(1 for x in P for q in Q if chi(x, (q, 0)) != chi_q(q, x[:2]))
print(len(P), "primary primes,", len(Q), "rational primes; reciprocity failures:", bad)

def is_x2_27y2(p):
    return any(math.isqrt(p - 27*y*y)**2 == p - 27*y*y for y in range(math.isqrt(p // 27) + 1))
bad = sum(1 for p in range(7, 10**5, 3) if p % 3 == 1 and pari.isprime(p)
          and (pow(2, (p - 1) // 3, p) == 1) != is_x2_27y2(p))
print("Gauss's theorem, failures below 10^5:", bad)

### P10.2
**How to find it.** Take a primitive root $g$ modulo $p$ and $\chi(g^k)=\omega^k$; build a table of discrete logarithms, and add up $\omega^{k(t)+k(1-t)}$ as a pair $(a,b)$. Then $J=a+b\omega$, and $4p=4N(J)=(2a-b)^2+3b^2$ with $3\mid b$ gives $4p=L^2+27M^2$, $L=2a-b$, $M=b/3$.

In [ ]:
W = [(1, 0), (0, 1), (-1, -1)]
rows = []
for p in range(7, 500, 3):
    if p % 3 != 1 or not pari.isprime(p): continue
    g = int(pari.znprimroot(p)); log = {}
    x = 1
    for k in range(p - 1):
        log[x] = k; x = x * g % p
    a = b = 0
    for t in range(2, p):
        w = W[(log[t] + log[(1 - t) % p]) % 3]
        a += w[0]; b += w[1]
    norm_ok = a*a - a*b + b*b == p
    primary = a % 3 == 2 and b % 3 == 0
    # chi = chi_pi for pi = J: check chi(g) = w via g^((p-1)/3) = w mod J
    r = -a * pow(b, -1, p) % p
    same = pow(g, (p - 1) // 3, p) == r
    rows.append((p, (a, b), norm_ok, primary, same, (2*a - b, b // 3)))
print(all(r[2] and r[3] and r[4] for r in rows), rows[:6])

All three properties hold for every $p$. The pair $(L,M)$ is Gauss's: for $p=31$, $J=5+6\omega$, so $L=2\cdot5-6=4$, $M=2$ and $4\cdot31=4^2+27\cdot2^2$. Gauss computed the number of solutions of $x^3+y^3\equiv1\pmod p$ from $L$; this is the first appearance of what Chapter 13 calls the Riemann hypothesis for curves over finite fields ($|J|=\sqrt p$).

### P10.3
**How to find it.** `quadhilbert(D)` returns a polynomial whose roots generate the Hilbert class field over $\mathbb Q(\sqrt D)$; `rnfconductor(bnfinit(K), pol)` returns the conductor of the abelian extension (trivial when unramified). `qfbsolve` or a direct search decides $p=x^2+ny^2$.

In [ ]:
for D in (-20, -56, -23):
    print(D, "h =", pari.qfbclassno(D), " Hilbert class field:", pari.quadhilbert(D))
cond = pari("rnfconductor(bnfinit(y^2+14), x^4+2*x^2-7)")
print("conductor of K(sqrt(2 sqrt2 - 1))/K:", cond[0], "(trivial ideal: unramified)")

def rep(p, a, b, c):
    for y in range(0, math.isqrt(4 * a * p // (4 * a * c - b * b)) + 2):
        # a x^2 + b x y + c y^2 = p: solve for x
        disc = b * b * y * y - 4 * a * (c * y * y - p)
        if disc >= 0 and math.isqrt(disc)**2 == disc:
            for s in (1, -1):
                num = -b * y + s * math.isqrt(disc)
                if num % (2 * a) == 0: return True
    return False

f14 = pari("x^4+2*x^2-7"); f23 = pari("x^3-x-1")
bad = Counter()
for p in pari.primes(10**4)[3:]:
    p = int(p)
    if p not in (5,):
        bad[5] += rep(p, 1, 0, 5) != (p % 20 in (1, 9))
    if p != 7:
        bad[14] += rep(p, 1, 0, 14) != (pari.kronecker(-14, p) == 1 and len(pari.polrootsmod(f14, p)) > 0)
    if p != 23:
        bad[23] += rep(p, 1, 1, 6) != (len(pari.polrootsmod(f23, p)) == 3)
print("failures:", dict(bad))
print("first p = x^2+xy+6y^2:", [int(p) for p in pari.primes(400) if int(p) != 23 and rep(int(p), 1, 1, 6)][:6])

No failures. (The loop stops at $10^4$ to keep the notebook fast; raising the bound to $10^5$ changes nothing.) The first primes of the form $x^2+xy+6y^2$ are $59, 101, 167, 173, 211, 223$: these are exactly the primes for which $X^3-X-1$ splits completely, the famous example where the "Frobenius trace" $a_p$ of a weight-one modular form ($q\prod(1-q^n)(1-q^{23n})$) equals $2$.

### P10.4
**How to find it.** The conductor divides a product of powers of the ramified primes; for a candidate $m$, $K\subseteq\mathbb Q(\zeta_m)$ iff the defining polynomial has a root in $\mathbb Q(\zeta_m)$ (`nfroots(nfinit(polcyclo(m)), f)`). For the eta product, expand the product as a power series with integer coefficients.

In [ ]:
def conductor(f):
    f = pari(f); d = int(pari.nfdisc(f))
    ram = [int(q) for q in pari.factor(abs(d))[0]]
    cands = sorted({math.prod(q**e for q, e in zip(ram, es))
                    for es in __import__('itertools').product(*[range(1, 5) for _ in ram])})
    for m in cands:
        if pari.eulerphi(m) % pari.poldegree(f) == 0 and len(pari.nfroots(pari.nfinit(pari.polcyclo(m, pari('y'))), f)) > 0:
            return m, d
for f in ["x^3-3*x+1", "x^3+x^2-4*x+1", "x^4-5*x^2+5", "x^4+x^3+x^2+x+1", "x^2-x+5"]:
    m, d = conductor(f)
    print(f"{f:18s} disc {d:>8d}   conductor {m}")

The conductors are $9, 13, 20, 5, 19$. For the cyclic cubic fields $d=m^2$ and for the quadratic field $|d|=m$ (Exercise 10.4): this is the *conductor-discriminant formula*, $|d_K|=\prod_\chi f_\chi$ over the characters of $\operatorname{Gal}(K/\mathbb Q)$, each with its own conductor ($X^4-5X^2+5$: characters of conductors $1,5,20,20$, product $2000=d$).

In [ ]:
N = 5000
a = [0] * (N + 1); a[1] = 1
for n in range(1, N // 6 + 1):
    for step in (6 * n, 18 * n):
        if step > N: continue
        for k in range(N, step - 1, -1):
            a[k] -= a[k - step]
bad = 0
for p in pari(f'primes([5, {N}])'):
    p = int(p)
    roots = len(pari.polrootsmod(pari("x^3-2"), p))
    trace = {3: 2, 1: 0, 0: -1}[roots]
    bad += a[p] != trace
print("eta(6z)eta(18z): a_p = trace of rho(Frob_p) for all p < 5000 except", bad, "primes")

### P10.5
**How to find it.** Step 1 is a random search: about $(\ln p)^2/4$ tries, each two cheap pseudo-primality tests. Step 2 calls `polclass` once (it does not depend on $p$) and `polrootsmod`. Step 3 uses that $y^2=x^3+3kx+2k$ has $j$-invariant $1728\cdot\frac{4(3k)^3}{4(3k)^3+27(2k)^2}=j$ when $k=\frac{j}{1728-j}$, and that the twist by a non-residue $c$, $y^2=x^3+3kc^2x+2kc^3$, has $2p+2-\#E$ points.

In [ ]:
def cm_curve(D, bits):
    aD = -D
    while True:
        t = random.getrandbits(bits // 2 + 1) | 1 | (1 << (bits // 2))
        v = random.getrandbits(bits // 2 - 4) | 1
        p = (t * t + aD * v * v) // 4
        if pari.ispseudoprime(p) and pari.ispseudoprime(p + 1 - t):
            break
    H = pari.polclass(D)
    roots = pari.polrootsmod(H, p)
    j = int(pari.lift(roots[0]))
    k = j * pow(1728 - j, -1, p) % p
    E = pari.ellinit([3 * k % p, 2 * k % p], p)
    if int(pari.ellcard(E)) != p + 1 - t:
        c = next(c for c in range(2, p) if pari.kronecker(c, p) == -1)
        E = pari.ellinit([3 * k * c * c % p, 2 * k * c**3 % p], p)
    return p, t, E, len(roots), int(pari.poldegree(H))

for D in (-83, -131, -251):
    t0 = time.time()
    p, t, E, nroots, h = cm_curve(D, 128)
    N = int(pari.ellcard(E))
    print(f"D={D}: h={h}, H_D has {nroots} roots mod p (splits completely: {nroots == h});"
          f" p has {p.bit_length()} bits, #E = p+1-t is prime: {N == p + 1 - t and bool(pari.isprime(N))};"
          f" {time.time() - t0:.2f}s")

In [ ]:
t0 = time.time(); tries = 0
p = int(pari.nextprime(2**95 + random.getrandbits(90)))  # 96 bits only: 128 would take minutes
while True:
    tries += 1
    E = pari.ellinit([random.randrange(p), random.randrange(p)], p)
    if pari.ispseudoprime(pari.ellcard(E)):
        break
print(f"naive method at 96 bits: {tries} random curves (point counting each), {time.time() - t0:.1f}s")

Even at $96$ bits the naive search is slower than the CM method at $128$ bits (at $128$ bits it takes about two minutes on a laptop), and the gap widens with the size of $p$, because it never counts points on a curve whose order it does not already know. The price is that the curve has complex multiplication by an order of small class number, a special structure that cryptographers watch carefully (no attack exploiting it is known for well-chosen parameters, but standards such as the NIST and Brainpool curves preferred "random" curves partly for this reason). The method is a direct application of class field theory: the Hilbert class polynomial generates the Hilbert class field, primes of the form $\frac{t^2+|D|v^2}4$ split completely in it (Corollary 10.14), and the Frobenius of the reduced CM curve is the element $\frac{t+v\sqrt D}2$ of norm $p$. This is Kronecker's Jugendtraum at work in your browser's TLS stack.